In [ ]:

!pip install -q -U langchain langchain-groq gspread

import os
import csv
import gspread

from typing import Literal
from pydantic import BaseModel, Field
from google.colab import userdata, auth
from google.auth import default

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, AIMessage


# 1. GROQ SETUP

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("Add GROQ_API_KEY to Google Colab Secrets.")

os.environ["GROQ_API_KEY"] = api_key

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_retries=1
)

print("Groq connected successfully.")


# 2. GOOGLE SHEETS SETUP

SHEET_NAME = "Smart Grocery Orders"
ws = None
sheet_url = None

print("\nConnecting to Google Sheets...")

try:
    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    try:
        sh = gc.open(SHEET_NAME)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_NAME)

    ws = sh.sheet1
    sheet_url = sh.url

    if not ws.get_all_values():
        ws.append_row([
            "Customer Name",
            "Product",
            "Quantity",
            "Unit",
            "Requested Date",
            "Total Price"
        ])

    print("Google Sheets connected successfully.")
    print("Sheet URL:", sheet_url)

except Exception as error:
    print("Google Sheets authentication failed.")
    print("Orders will be saved in a local CSV file for now.")
    print("Reason:", str(error))


# 3. PRODUCT AND STORE DETAILS

PRODUCTS = {
    "rice": {"price": 60, "unit": "kg"},
    "sugar": {"price": 45, "unit": "kg"},
    "milk": {"price": 30, "unit": "litre"},
    "bread": {"price": 40, "unit": "packet"},
    "eggs": {"price": 7, "unit": "piece"},
    "apples": {"price": 120, "unit": "kg"},
    "tomatoes": {"price": 35, "unit": "kg"},
    "potatoes": {"price": 30, "unit": "kg"},
    "oil": {"price": 150, "unit": "litre"}
}

HOURS = {
    "monday": "9 AM to 8 PM",
    "tuesday": "9 AM to 8 PM",
    "wednesday": "9 AM to 8 PM",
    "thursday": "9 AM to 8 PM",
    "friday": "9 AM to 8 PM",
    "saturday": "9 AM to 8 PM",
    "sunday": "10 AM to 6 PM"
}

print("\nAvailable products:")

for item, details in PRODUCTS.items():
    print(
        f"{item.title()} - Rs.{details['price']} per {details['unit']}"
    )


# 4. PYDANTIC ORDER MODEL

class Order(BaseModel):
    name: str = Field(description="Customer's first name")

    item: Literal[
        "rice", "sugar", "milk", "bread", "eggs",
        "apples", "tomatoes", "potatoes", "oil"
    ]

    qty: int = Field(ge=1, description="Required quantity")

    date: str = Field(
        description="Requested delivery or pickup date"
    )


print("\nTesting structured order extraction...")

try:
    extractor = llm.with_structured_output(Order)

    sample = (
        "Hi, I am Anusha. I need 2 kg rice "
        "for 24 December."
    )

    extracted_order = extractor.invoke(sample)

    print(extracted_order.model_dump())

except Exception as error:
    print("Structured extraction test could not run:", error)


# 5. PYDANTIC VALIDATION

try:
    Order(
        name="Anusha",
        item="rice",
        qty=0,
        date="24 December"
    )
except Exception:
    print("\nInvalid quantity rejected successfully.")


# 6. GROCERY TOOLS

@tool
def get_price(item: str, qty: int = 1) -> str:
    """Get the total price of a grocery product for the requested quantity."""

    item = item.lower().strip()

    if qty < 1:
        return "Quantity must be at least 1."

    product = PRODUCTS.get(item)

    if product is None:
        return (
            f"{item} is not in our catalogue. "
            f"Available products: {', '.join(PRODUCTS.keys())}."
        )

    total = product["price"] * qty

    return (
        f"{qty} {product['unit']} of {item} "
        f"costs Rs.{total}."
    )


@tool
def is_open(day: str) -> str:
    """Check the store opening hours for a particular weekday."""

    day = day.lower().strip()
    hours = HOURS.get(day)

    if hours is None:
        return "Please provide a valid day from Monday to Sunday."

    return f"Our store timings on {day.title()} are {hours}."


@tool
def log_order(name: str, item: str, qty: int, date: str) -> str:
    """Save an order only after the customer confirms placing it and provides all required details."""

    try:
        order = Order(
            name=name.strip(),
            item=item.lower().strip(),
            qty=qty,
            date=date.strip()
        )

        product = PRODUCTS[order.item]
        unit = product["unit"]
        total = product["price"] * order.qty

        row = [
            order.name,
            order.item,
            order.qty,
            unit,
            order.date,
            total
        ]

        if ws is not None:
            ws.append_row(row, value_input_option="USER_ENTERED")
            storage = "Google Sheets"
        else:
            file_exists = os.path.exists("grocery_orders.csv")

            with open(
                "grocery_orders.csv",
                "a",
                newline="",
                encoding="utf-8"
            ) as file:
                writer = csv.writer(file)

                if not file_exists:
                    writer.writerow([
                        "Customer Name",
                        "Product",
                        "Quantity",
                        "Unit",
                        "Requested Date",
                        "Total Price"
                    ])

                writer.writerow(row)

            storage = "local CSV file"

        return (
            f"Order saved successfully in {storage}. "
            f"Customer: {order.name}. "
            f"Product: {order.qty} {unit} of {order.item}. "
            f"Date: {order.date}. Total: Rs.{total}."
        )

    except Exception as error:
        return f"Could not save the order: {str(error)}"


print("\nAll three grocery tools are ready.")


# 7. LANGCHAIN AGENT

SYSTEM = """
You are a friendly Smart Grocery Shopping Assistant.

You help customers with product prices, store timings and grocery orders.

Product catalogue:
Rice: Rs.60 per kg
Sugar: Rs.45 per kg
Milk: Rs.30 per litre
Bread: Rs.40 per packet
Eggs: Rs.7 per piece
Apples: Rs.120 per kg
Tomatoes: Rs.35 per kg
Potatoes: Rs.30 per kg
Oil: Rs.150 per litre

Rules:

1. Always use get_price for product price questions.
2. Always use is_open for store timing questions.
3. Never invent prices.
4. The catalogue does not show live stock.
5. Never promise delivery.
6. For orders, collect the customer's name, product, quantity and requested date.
7. Ask for missing details before placing an order.
8. Call log_order only after the customer clearly confirms placing the order.
9. Never invent customer details.
10. Use the tools to calculate prices.
11. Confirm the saved order details and total price.
12. Respond in English or Telugu according to the customer's language.
13. Keep responses short, clear and friendly.

The store timings and product prices are sample information.
"""

agent = create_agent(
    model=llm,
    tools=[get_price, is_open, log_order],
    system_prompt=SYSTEM
)

print("LangChain grocery agent created successfully.")


# 8. CHAT FUNCTION

history = []
MAX_HISTORY = 8

def chat(text):
    global history

    try:
        result = agent.invoke({
            "messages": history + [
                HumanMessage(content=text)
            ]
        })

        response = result["messages"][-1].content

        if not isinstance(response, str):
            response = str(response)

        history.extend([
            HumanMessage(content=text),
            AIMessage(content=response)
        ])

        history = history[-(MAX_HISTORY * 2):]

        return response

    except Exception as error:
        print("Error:", str(error))
        return "Sorry, something went wrong. Please try again."


# 9. FIRST AGENT TEST

print("\nAgent test:")

question = "What is the price of 2 kg rice?"

print("Customer:", question)
print("Assistant:", chat(question))


# 10. INTERACTIVE CHATBOT

print("\n" + "=" * 50)
print("SMART GROCERY SHOPPING ASSISTANT")
print("=" * 50)

print("Ask about prices, store timings or place an order.")
print("Type 'exit' to stop.")

while True:
    user_input = input("\nYou: ").strip()

    if user_input.lower() == "exit":
        print("Thank you for using Smart Grocery Shopping Assistant!")
        break

    if not user_input:
        print("Please enter a message.")
        continue

    print("\nAssistant:", chat(user_input))


# 11. ORDER RECORDS

print("\nOrder storage:")

if ws is not None:
    print("Google Sheets:", sheet_url)
else:
    print("Local order file: grocery_orders.csv")


# 12. DAY 5 PREPARATION

business_facts = """
SMART GROCERY SHOPPING ASSISTANT

Product prices:
Rice: Rs.60 per kg
Sugar: Rs.45 per kg
Milk: Rs.30 per litre
Bread: Rs.40 per packet
Eggs: Rs.7 per piece
Apples: Rs.120 per kg
Tomatoes: Rs.35 per kg
Potatoes: Rs.30 per kg
Oil: Rs.150 per litre

Store timings:
Monday to Saturday: 9 AM to 8 PM
Sunday: 10 AM to 6 PM

The assistant can:
1. Answer grocery price questions.
2. Calculate product totals.
3. Provide store timings.
4. Collect grocery order details.
5. Validate orders using Pydantic.
6. Save confirmed orders to Google Sheets or a local CSV file.

Prices and timings are sample information.
The catalogue does not provide live stock information.
"""

with open("business_facts.txt", "w", encoding="utf-8") as file:
    file.write(business_facts)

print("\nCreated business_facts.txt")
print("Day 4 execution completed.")


Groq connected successfully.

Connecting to Google Sheets...
Google Sheets connected successfully.
Sheet URL: https://docs.google.com/spreadsheets/d/1kfRm7kfAQs1LtP-dwDwbrVRQH9-cFRGnTOrThr1z9hQ

Available products:
Rice - Rs.60 per kg
Sugar - Rs.45 per kg
Milk - Rs.30 per litre
Bread - Rs.40 per packet
Eggs - Rs.7 per piece
Apples - Rs.120 per kg
Tomatoes - Rs.35 per kg
Potatoes - Rs.30 per kg
Oil - Rs.150 per litre

Testing structured order extraction...
{'name': 'Anusha', 'item': 'rice', 'qty': 2, 'date': '2023-12-24'}

Invalid quantity rejected successfully.

All three grocery tools are ready.
LangChain grocery agent created successfully.

Agent test:
Customer: What is the price of 2 kg rice?
Assistant: 2 kg of rice costs Rs. 120.

SMART GROCERY SHOPPING ASSISTANT
Ask about prices, store timings or place an order.
Type 'exit' to stop.

You: how much is one kg potatoes?

Assistant: One kilogram of potatoes is Rs. 30.

You: do u have oil packets

Assistant: I don’t have live stock inf

In [ ]:
headers = [
    "Customer Name",
    "Product",
    "Quantity",
    "Unit",
    "Requested Date",
    "Total Price"
]

ws.insert_row(headers, 1)



{'spreadsheetId': '1kfRm7kfAQs1LtP-dwDwbrVRQH9-cFRGnTOrThr1z9hQ',
 'updates': {'spreadsheetId': '1kfRm7kfAQs1LtP-dwDwbrVRQH9-cFRGnTOrThr1z9hQ',
  'updatedRange': 'Sheet1!A1:F1',
  'updatedRows': 1,
  'updatedColumns': 6,
  'updatedCells': 6}}